In [1]:
import copy
import random
import time
import math
import numpy as np
import pandas as pd

# Your project imports
from tsn_scheduling_problem import TSNScheduling
from schedule import Schedule, Slot

# Benchmark utilities you already have
from benchmark_utils import (
    build_network_generator_from_usecase,
    generate_one_arrival_stream,
    build_usecase_stream_from_request,
)

# Optional: cleaner printing
np.set_printoptions(suppress=True, linewidth=140)

print("Imports loaded.")

Imports loaded.


In [19]:
CONFIG_PATH = "uc_config.cfg"
INSTANCE_PATH = "finalmotawassit.json"   # change if you want another instance

problem = TSNScheduling(CONFIG_PATH)
instance = problem.import_scheduling_usecase(INSTANCE_PATH)

# Keep problem aligned with this instance
problem.usecase = instance
problem.build_metadata()

print("Loaded instance:", INSTANCE_PATH)
print("Number of streams in usecase:", len(instance.streams))
print("Number of scheduled slots:", len(instance.schedule))

active_stream_ids = sorted({
    int(slot.stream_id)
    for slot in instance.schedule
    if slot.stream_id is not None
})

print("Active scheduled streams:", len(active_stream_ids))
print("Active stream IDs:", active_stream_ids)
print("Cycle ns:", instance.schedule.cycle_ns)

Loaded instance: finalmotawassit.json
Number of streams in usecase: 48
Number of scheduled slots: 152
Active scheduled streams: 38
Active stream IDs: [1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16, 17, 18, 19, 20, 22, 24, 25, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 39, 40, 41, 42, 44, 45, 46, 47]
Cycle ns: 1000000


In [3]:
from evaluator import TSNConstraintEvaluator, ScheduleEncodingType

def check_schedule_validity(instance, problem, label="schedule"):
    """
    Check whether the current instance.schedule has evaluator violations.
    """
    problem.usecase = instance
    problem.build_metadata()

    evaluator = TSNConstraintEvaluator(
        problem,
        check_routing=False,
        check_task_placement=False,
        scheduling_type=ScheduleEncodingType.START_TIME,
    )

    compact = instance.schedule.to_compact_encoding(
        problem.n_ports,
        problem.n_streams,
    )

    violations = evaluator.per_stream_all_violations(compact)

    breakdown = {
        "seq": int(violations[0].sum()) if violations.shape[0] > 0 else 0,
        "fifo": int(violations[1].sum()) if violations.shape[0] > 1 else 0,
        "macrocycle": int(violations[2].sum()) if violations.shape[0] > 2 else 0,
        "talker_offset": int(violations[3].sum()) if violations.shape[0] > 3 else 0,
        "latency": int(violations[4].sum()) if violations.shape[0] > 4 else 0,
        "total": int(violations.sum()),
    }

    print(f"[{label}] violation breakdown:", breakdown)
    return breakdown

validity = check_schedule_validity(instance, problem, label="initial")

[initial] violation breakdown: {'seq': 0, 'fifo': 66, 'macrocycle': 40, 'talker_offset': 0, 'latency': 0, 'total': 106}


In [4]:
import os
import copy
import numpy as np
from collections import defaultdict

from tsn_scheduling_problem import TSNScheduling
from schedule import Schedule, Slot
from evaluator import TSNConstraintEvaluator, ScheduleEncodingType


def rebuild_schedule_guided_by_old_offsets(
    config_path="/home/ws/TSN_optimizer_platform/uc_config.cfg",
    instance_path="/home/ws/TSN_optimizer_platform/s.json",
    output_path="/home/ws/TSN_optimizer_platform/final/small_valid.json",
    queue_id=5,
    max_candidates_per_hop=12,
    beam_width=120,
    ordering="old_offset",
    verbose=True,
):
    """
    Full reconfiguration, but guided by the old MILP schedule.

    Difference from pure first-fit:
    - We delete the whole schedule.
    - We re-place streams again.
    - For each stream, old hop starts are used as preferred anchors.
    - If old downstream hop is too early according to evaluator timing,
      it is pushed forward.
    - We avoid overlaps using periodic interval checks.
    - We validate the whole accepted schedule after every placement.
    """

    # ------------------------------------------------------------
    # Load problem/usecase/schedule
    # ------------------------------------------------------------
    problem = TSNScheduling(config_path)
    usecase = problem.import_scheduling_usecase(instance_path)

    problem.usecase = usecase
    problem.build_metadata()

    port_name_to_id, port_id_to_name = usecase.port_name_id_mapping
    old_schedule = Schedule.from_usecase(usecase, port_name_to_id)

    cycle_ns = int(old_schedule.cycle_ns)

    evaluator = TSNConstraintEvaluator(
        problem,
        check_routing=False,
        check_task_placement=False,
        scheduling_type=ScheduleEncodingType.START_TIME,
    )

    active_stream_ids = sorted({
        int(slot.stream_id)
        for slot in old_schedule
        if slot.stream_id is not None
    })

    if verbose:
        print(f"Loaded: {instance_path}")
        print(f"cycle_ns={cycle_ns}")
        print(f"n_ports={problem.n_ports}, n_streams={problem.n_streams}")
        print(f"old slots={len(old_schedule)}")
        print(f"active streams={len(active_stream_ids)}")

    # ------------------------------------------------------------
    # Old starts by stream and port
    # ------------------------------------------------------------
    old_start_by_stream_port = {}
    old_first_start = {}

    for sid in active_stream_ids:
        old_start_by_stream_port[sid] = {}

        slots = [
            slot for slot in old_schedule
            if slot.stream_id is not None and int(slot.stream_id) == int(sid)
        ]

        for slot in slots:
            old_start_by_stream_port[sid][int(slot.port_id)] = int(slot.start_ns)

        starts = [int(slot.start_ns) for slot in slots]
        old_first_start[sid] = min(starts) if starts else 10**18

    def stream_hardness(sid):
        s = usecase.streams[int(sid)]
        period = int(s.get("interval", cycle_ns))
        latency = int(s.get("maxLatency", period))
        frame = int(s.get("maxFrameSize", 0))

        path_len = 0
        if "pathLists" in s and s["pathLists"]:
            path_len = max(0, len(s["pathLists"][0]) - 1)

        tightness = period / max(1, latency)

        return (
            path_len,
            tightness,
            frame,
            -period,
            -old_first_start.get(sid, 10**18),
            sid,
        )

    if ordering == "old_offset":
        ordered_stream_ids = sorted(
            active_stream_ids,
            key=lambda sid: (old_first_start.get(sid, 10**18), sid)
        )
    elif ordering == "hard_first":
        ordered_stream_ids = sorted(
            active_stream_ids,
            key=lambda sid: stream_hardness(sid),
            reverse=True
        )
    else:
        raise ValueError("ordering must be 'old_offset' or 'hard_first'")

    if verbose:
        print("ordering:", ordering)
        print("first 20:", ordered_stream_ids[:20])

    # ------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------
    def periodic_intervals_for_slot(slot, span_ns):
        out = []
        k = 0
        while int(slot.start_ns) + k * int(slot.period_ns) < int(span_ns):
            a = int(slot.start_ns) + k * int(slot.period_ns)
            b = int(slot.end_ns) + k * int(slot.period_ns)
            if b <= int(span_ns):
                out.append((a, b))
            k += 1
        return out

    def refresh_port_intervals(schedule):
        port_intervals = {}
        for slot in schedule:
            if slot.port_id is None:
                continue
            pid = int(slot.port_id)
            port_intervals.setdefault(pid, [])
            port_intervals[pid].extend(periodic_intervals_for_slot(slot, schedule.cycle_ns))

        for pid in port_intervals:
            port_intervals[pid] = sorted(port_intervals[pid])

        return port_intervals

    def fits_periodic(start, duration, period, intervals, cycle_ns):
        start = int(start)
        duration = int(duration)
        period = int(period)
        cycle_ns = int(cycle_ns)

        if start < 0 or start + duration > period:
            return False

        k = 0
        while start + k * period < cycle_ns:
            a = start + k * period
            b = a + duration

            if b > cycle_ns:
                return False

            for x, y in intervals:
                if a < y and b > x:
                    return False

            k += 1

        return True

    def candidate_starts_near_old(intervals, min_start, old_start, duration, period, limit):
        """
        Generate candidate starts close to the old MILP start, but not earlier than evaluator min_start.
        """
        min_start = int(max(0, min_start))
        old_start = int(max(0, old_start))
        duration = int(duration)
        period = int(period)

        latest_start = int(period - duration)
        if latest_start < min_start:
            return []

        seeds = set()

        # Main anchors
        seeds.add(max(min_start, old_start))
        seeds.add(min_start)

        # Around old start
        for d in [0, 500, 1000, 2000, 4000, 8000, 12000, 16000, 24000, 32000]:
            seeds.add(max(min_start, old_start + d))
            seeds.add(max(min_start, old_start - d))

        # Right after existing occupied intervals
        for a, b in intervals:
            seeds.add(max(min_start, int(b)))
            seeds.add(max(min_start, int(a) - duration))

        # Use _first_non_overlapping_start to jump to valid starts
        starts = []

        for seed in sorted(seeds, key=lambda x: abs(int(x) - old_start)):
            if seed < min_start or seed > latest_start:
                continue

            cand = new_schedule._first_non_overlapping_start(
                intervals=intervals,
                min_start=int(seed),
                duration=int(duration),
                period_ns=int(period),
            )

            if cand is None:
                continue

            cand = int(cand)

            if cand < min_start or cand > latest_start:
                continue

            if fits_periodic(cand, duration, period, intervals, cycle_ns):
                if cand not in starts:
                    starts.append(cand)

            if len(starts) >= int(limit):
                break

        return starts[:limit]

    def full_schedule_valid(schedule):
        conflicts = schedule.validate_conflicts()
        if conflicts:
            return False, conflicts, None

        try:
            compact = schedule.to_compact_encoding(problem.n_ports, problem.n_streams)
            violations = evaluator.per_stream_all_violations(compact)
            ok = bool(np.all(violations == 0))
            return ok, conflicts, violations
        except Exception as e:
            return False, conflicts, str(e)

    def try_place_stream_guided(schedule, sid, port_intervals):
        sid = int(sid)
        stream = usecase.streams[sid]

        if "pathLists" not in stream or not stream["pathLists"]:
            return None

        path = stream["pathLists"][0]
        hop_port_names = path[:-1]
        hop_port_ids = [int(port_name_to_id[p]) for p in hop_port_names]

        if not hop_port_ids:
            return None

        period = int(stream["interval"])

        # Beam state:
        # slots, starts, score
        beam = [([], [], 0.0)]

        for hop_idx, pid in enumerate(hop_port_ids):
            duration = int(max(1, round(evaluator.occupiedLengths[pid - 1, sid])))

            new_beam = []

            for partial_slots, starts, _ in beam:
                if hop_idx == 0:
                    min_start = 0
                else:
                    prev_pid = hop_port_ids[hop_idx - 1]
                    prev_start = starts[-1]
                    min_start = prev_start + int(round(evaluator.all_hops_costs[prev_pid - 1, sid]))

                old_start = old_start_by_stream_port.get(sid, {}).get(pid, min_start)

                intervals = list(port_intervals.get(pid, []))

                # add already selected slots of this same candidate
                for s in partial_slots:
                    if int(s.port_id) == int(pid):
                        intervals.extend(periodic_intervals_for_slot(s, cycle_ns))

                intervals = sorted(intervals)

                cand_starts = candidate_starts_near_old(
                    intervals=intervals,
                    min_start=min_start,
                    old_start=old_start,
                    duration=duration,
                    period=period,
                    limit=max_candidates_per_hop,
                )

                for st in cand_starts:
                    slot = Slot(
                        start_ns=int(st),
                        end_ns=int(st + duration),
                        period_ns=int(period),
                        stream_id=int(sid),
                        port_id=int(pid),
                        queue_id=queue_id,
                    )

                    new_slots = partial_slots + [slot]
                    new_starts = starts + [int(st)]

                    # score: prefer staying close to old MILP starts, with slight preference for compactness
                    old_vec = [
                        old_start_by_stream_port.get(sid, {}).get(hop_port_ids[i], new_starts[i])
                        for i in range(len(new_starts))
                    ]

                    deviation = sum(abs(int(a) - int(b)) for a, b in zip(new_starts, old_vec))
                    span = max(new_starts) - min(new_starts) if new_starts else 0
                    first = new_starts[0] if new_starts else 0

                    score = deviation + 0.05 * span + 0.001 * first
                    new_beam.append((new_slots, new_starts, score))

            if not new_beam:
                return None

            new_beam.sort(key=lambda x: x[2])
            beam = new_beam[:beam_width]

        # Validate complete candidate globally
        base_slots = list(schedule)

        for slots, starts, score in sorted(beam, key=lambda x: x[2]):
            tmp = Schedule(
                cycle_ns=schedule.cycle_ns,
                basetime=schedule.basetime,
                iterable=base_slots + list(slots),
            )

            ok, conflicts, violations = full_schedule_valid(tmp)
            if ok:
                return {
                    "slots": slots,
                    "starts": starts,
                    "score": score,
                }

        return None

    # ------------------------------------------------------------
    # Full rebuild
    # ------------------------------------------------------------
    # Keep pathLists; clear only old offset/latency annotations.
    for sid in active_stream_ids:
        s = usecase.streams[int(sid)]
        for key in ("offset", "accLatency"):
            if key in s:
                del s[key]

    usecase.removeSchedule()

    new_schedule = Schedule(cycle_ns=cycle_ns, basetime=old_schedule.basetime)
    port_intervals = {}

    accepted = []
    rejected = []

    best_schedule = copy.deepcopy(new_schedule)
    best_accepted = []

    for k, sid in enumerate(ordered_stream_ids, start=1):
        candidate = try_place_stream_guided(new_schedule, sid, port_intervals)

        if candidate is None:
            rejected.append(int(sid))
            if verbose:
                print(f"[{k:03d}/{len(ordered_stream_ids)}] rejected stream {sid}")
            continue

        for slot in candidate["slots"]:
            new_schedule.append(slot)

        new_schedule.sort()
        port_intervals = refresh_port_intervals(new_schedule)

        accepted.append(int(sid))
        best_schedule = copy.deepcopy(new_schedule)
        best_accepted = list(accepted)

        if verbose:
            print(
                f"[{k:03d}/{len(ordered_stream_ids)}] accepted stream {sid} "
                f"starts={candidate['starts']} score={candidate['score']:.1f}"
            )

    # ------------------------------------------------------------
    # Save final schedule
    # ------------------------------------------------------------
    new_schedule = best_schedule

    usecase.removeSchedule()
    usecase.schedule = new_schedule
    _, port_id_to_name = usecase.port_name_id_mapping
    new_schedule.update_graph(usecase.graph, port_id_to_name)

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    usecase.to_file(
        jsnPath=os.path.dirname(output_path),
        ucFileName=os.path.basename(output_path),
    )

    ok, conflicts, violations = full_schedule_valid(new_schedule)

    if isinstance(violations, np.ndarray):
        row_sums = np.sum(violations != 0, axis=1)
        col_sums = np.sum(violations != 0, axis=0)
        violating_streams = np.where(col_sums > 0)[0].tolist()
    else:
        row_sums = None
        violating_streams = []

    print("\n================ GUIDED REBUILD SUMMARY ================")
    print(f"Accepted streams: {len(best_accepted)}")
    print(f"Rejected streams: {len(active_stream_ids) - len(best_accepted)}")
    print(f"Accepted IDs: {best_accepted}")
    print(f"Conflict pairs: {len(conflicts)}")
    print(f"Violation row sums: {row_sums}")
    print(f"Violating streams: {violating_streams}")
    print(f"VALID: {ok}")
    print(f"Saved to: {output_path}")
    print("========================================================")

    return {
        "valid": ok,
        "accepted": best_accepted,
        "rejected": [sid for sid in active_stream_ids if sid not in best_accepted],
        "schedule": new_schedule,
        "usecase": usecase,
        "violations": violations,
        "output_path": output_path,
    }


In [18]:
result = rebuild_schedule_guided_by_old_offsets(
    config_path="/home/ws/TSN_optimizer_platform/uc_config.cfg",
    instance_path="/home/ws/TSN_optimizer_platform/motawassit.json",
    output_path="/home/ws/TSN_optimizer_platform/final/motawassit.json",
    queue_id=5,
    max_candidates_per_hop=1000,
    beam_width=100000,
    ordering="hard_first",  # "old_offset" or "hard_first"
    verbose=True,
)

Loaded: /home/ws/TSN_optimizer_platform/motawassit.json
cycle_ns=1000000
n_ports=32, n_streams=48
old slots=192
active streams=48
ordering: hard_first
first 20: [33, 40, 18, 6, 42, 39, 17, 35, 13, 5, 43, 9, 11, 29, 46, 2, 44, 24, 10, 25]
[001/48] accepted stream 33 starts=[296000, 304000, 312000, 320000] score=1496.0
[002/48] accepted stream 40 starts=[38000, 46000, 54000, 62000] score=1238.0
[003/48] accepted stream 18 starts=[312000, 320000, 328000, 336000] score=1512.0
[004/48] accepted stream 6 starts=[15000, 21000, 27000, 33000] score=915.0
[005/48] accepted stream 42 starts=[70000, 76000, 82000, 88000] score=970.0
[006/48] accepted stream 39 starts=[275000, 281000, 287000, 293000] score=1175.0
[007/48] accepted stream 17 starts=[0, 5000, 10000, 15000] score=750.0
[008/48] accepted stream 35 starts=[289000, 294000, 299000, 304000] score=1039.0
[009/48] rejected stream 13
[010/48] accepted stream 5 starts=[294488, 300488, 310000, 322000] score=134694.1
[011/48] rejected stream 43
[